# Two-term Boltzmann solver

Fluid and global models need the electron transport coefficients and rate
coefficients as functions of the reduced field $E/N$ or of the mean electron
energy. `solve_eedf` computes them from the same `CollisionProcess` objects the
Monte Carlo collisions use, so that every model of a discharge works with rates
consistent with one cross-section set.

It solves the classical two-term approximation of the Boltzmann equation for
electrons in a uniform DC field, following Hagelaar and Pitchford, Plasma
Sources Sci. Technol. 14, 722 (2005), the method of BOLSIG+. The energy
distribution $F_0(\varepsilon)$, normalised by
$\int \varepsilon^{1/2} F_0\, d\varepsilon = 1$, obeys

$$\frac{d}{d\varepsilon}\left(\tilde W F_0 - \tilde D \frac{dF_0}{d\varepsilon}\right) = \tilde S,
\qquad \tilde W = -\gamma\varepsilon^2\sigma_\varepsilon,
\qquad \tilde D = \frac{\gamma}{3}\left(\frac{E}{N}\right)^2\frac{\varepsilon}{\tilde\sigma_m} + \frac{\gamma k T}{e}\,\varepsilon^2\sigma_\varepsilon,$$

with $\gamma = (2e/m_e)^{1/2}$, $\sigma_\varepsilon = \sum_k x_k (2m_e/M_k)\sigma_k$ the
elastic energy-loss cross section, $\tilde\sigma_m$ the total momentum-transfer
cross section (plus the growth term) and $\tilde S$ the inelastic collisions:
excitation moves electrons down by the energy loss, ionization turns one
electron into two, attachment removes one.

In [ ]:
import matplotlib.pyplot as plt

# light-mode categorical palette, used in this fixed order
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2.0,
        "lines.markersize": 6,
        "legend.frameon": False,
        "figure.figsize": (7.0, 4.2),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.prop_cycle": plt.cycler(color=COLORS),
    }
)

## A model gas

The cross sections below are a **synthetic argon-like model gas**, not measured
data: an elastic momentum-transfer cross section, one excitation at 11.5 eV and
an ionization at 15.8 eV. They are written in LXCat format and read back, the
path a real data set from [LXCat](https://lxcat.net) takes.

In [ ]:
from pathlib import Path

import numpy as np

import plasmacoll
from plasmacoll.constants import ATOMIC_MASS, ELECTRON_MASS, kelvin_to_ev

energy = np.round(np.geomspace(1e-2, 1e3, 120), 5)


def block(kind, target, parameter, sigma, process):
    lines = [kind, target, f" {parameter:.6e}", f"PROCESS: {process}", "-----"]
    lines += [f" {e:.6e}\t{s:.6e}" for e, s in zip(energy, sigma, strict=True)]
    return "\n".join([*lines, "-----"])


M = 40 * ATOMIC_MASS
x = np.clip(energy / 15.8, 1, None)
Path("model_gas.txt").write_text(
    "\n\n".join(
        [
            "SYNTHETIC MODEL GAS: these numbers are not measured data.",
            block(
                "ELASTIC",
                "Gas",
                ELECTRON_MASS / M,
                8e-20 / (1 + (energy / 10) ** 0.8),
                "E + Gas -> E + Gas, Elastic",
            ),
            block(
                "EXCITATION",
                "Gas -> Gas*",
                11.5,
                np.where(
                    energy > 11.5,
                    2e-21 * (energy - 11.5) / (1 + (energy / 30) ** 1.5),
                    0,
                ),
                "E + Gas -> E + Gas*, Excitation",
            ),
            block(
                "IONIZATION",
                "Gas -> Gas^+",
                15.8,
                np.where(energy > 15.8, 3e-20 * np.log(x) / x, 0),
                "E + Gas -> E + E + Gas+, Ionization",
            ),
        ]
    )
)
table = plasmacoll.read_lxcat("model_gas.txt")
Path("model_gas.txt").unlink()

gas = plasmacoll.NeutralBackground("gas", 1e22, kelvin_to_ev(300.0), M)
processes = [
    plasmacoll.CollisionProcess(
        "elastic", "gas", table[0].cross_section, name="elastic"
    ),
    plasmacoll.CollisionProcess(
        "excitation", "gas", table[1].cross_section, name="excitation"
    ),
    plasmacoll.CollisionProcess(
        "ionization",
        "gas",
        table[2].cross_section,
        name="ionization",
        products={"electron": "e", "ion": "ion"},
    ),
]

## The energy distribution at a few fields

`solve_eedf(processes, mixture, E/N)` takes $E/N$ in Td
($1\,\mathrm{Td} = 10^{-21}\,\mathrm{V\,m^2}$) and the gas temperature from the
mixture. It chooses the energy grid so that the distribution falls by 10 to 20
decades across it. Plotted is the energy probability function
$F_0 = f(\varepsilon)/\varepsilon^{1/2}$, on which a Maxwellian is a straight line:
above the excitation threshold the inelastic collisions deplete the tail.

In [ ]:
fig, ax = plt.subplots()
for color, reduced_field in zip(COLORS, (10, 30, 100, 300), strict=False):
    solution = plasmacoll.solve_eedf(processes, [gas], reduced_field)
    ax.semilogy(
        solution.energy,
        solution.eepf,
        color=color,
        label=f"{reduced_field} Td, <ε> = {solution.mean_energy:.1f} eV",
    )
ax.axvline(11.5, color=MUTED, linewidth=1, linestyle=":")
ax.set_xlim(0, 60)
ax.set_ylim(1e-7, 1)
ax.set_xlabel("electron energy (eV)")
ax.set_ylabel(r"EEPF $F_0$ (eV$^{-3/2}$)")
ax.set_title(
    "Inelastic collisions bend the distribution down above 11.5 eV", loc="left"
)
ax.legend(loc="upper right");

Besides the distribution, a solution has the mean energy, the reduced mobility
$\mu N$ and diffusion coefficient $DN$, the rate coefficient of every process
and the reduced Townsend coefficient $\alpha/N$. Its power balance compares the
power gained from the field with the power lost in collisions, and is a measure
of the discretisation error:

In [ ]:
solution = plasmacoll.solve_eedf(processes, [gas], 100.0)
print(f"mean energy      {solution.mean_energy:.3f} eV")
print(f"mu N             {solution.mobility:.4g} 1/(V m s)")
print(f"D N              {solution.diffusion:.4g} 1/(m s)")
print(f"drift velocity   {solution.drift_velocity:.4g} m/s")
for name, rate in solution.rate_coefficients.items():
    print(f"k {name:14s} {rate:.4g} m^3/s")
print(f"alpha/N          {solution.townsend_coefficient:.4g} m^2")
print(f"power balance    {solution.power_balance:.1e}")

## Rates against the mean energy

`swarm_table` solves over a range of fields and tabulates the results. Fluid
models look the coefficients up against the mean energy, which
`SwarmTable.interpolate` does, and `SwarmTable.rate_table` returns a rate as a
`RateTable` against the effective temperature $\tfrac23\langle\varepsilon\rangle$,
the form global models use.

In [ ]:
swarm = plasmacoll.swarm_table(processes, [gas], np.geomspace(0.3, 1000, 25))

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.6))
for color, name in zip(COLORS, ("elastic", "excitation", "ionization"), strict=False):
    rates = swarm.rate_coefficients[name]
    shown = rates > 1e-22
    left.loglog(swarm.mean_energy[shown], rates[shown], color=color, label=name)
left.set(xlabel="mean energy (eV)", ylabel="rate coefficient (m$^3$/s)", title="Rates")
left.legend()
right.loglog(swarm.mean_energy, swarm.mobility, color=COLORS[0], label=r"$\mu N$")
right.loglog(swarm.mean_energy, swarm.diffusion, color=COLORS[1], label=r"$DN$")
right.set(
    xlabel="mean energy (eV)",
    ylabel=r"1/(V m s) and 1/(m s)",
    title="Transport",
)
right.legend()
for ax in (left, right):
    ax.minorticks_off()
    ax.set_xticks([1, 2, 5, 10, 20], ["1", "2", "5", "10", "20"])
fig.tight_layout()

ionization = swarm.rate_table("ionization")
print(f"k_iz at <ε> = 6 eV:         {swarm.interpolate('ionization', 6.0):.3g} m^3/s")
print(f"RateTable at Te = 4 eV:     {ionization(4.0):.3g} m^3/s")
print(f"E/N for <ε> = 6 eV:         {swarm.interpolate('reduced_field', 6.0):.1f} Td")

## Checks against published results

Reid's ramp model gas (Aust. J. Phys. 32, 231 (1979)) is the standard
benchmark of Boltzmann solvers: a constant elastic cross section of
$6\,\text{Å}^2$, an excitation of $10(\varepsilon - 0.2)\,\text{Å}^2/\mathrm{eV}$
above 0.2 eV, $M = 4$ u and a cold gas. The reference values are the two-term
results of Flynn et al., J. Phys. D 57, 255204 (2024), Table 1.

In [ ]:
reid_gas = plasmacoll.NeutralBackground("gas", 1e22, 0.0, 4 * ATOMIC_MASS)
reid = [
    plasmacoll.CollisionProcess(
        "elastic", "gas", plasmacoll.CrossSection.constant(6e-20)
    ),
    plasmacoll.CollisionProcess(
        "excitation",
        "gas",
        plasmacoll.CrossSection([0.2, 1000.2], [0.0, 1e-16], threshold=0.2),
    ),
]
published = {  # E/N: mean energy (eV), drift velocity (m/s), D N (1/(m s))
    1: (0.1015, 1.2750e4, 0.9905e24),
    12: (0.2735, 7.0290e4, 1.3690e24),
    24: (0.4162, 9.1447e4, 1.4382e24),
}
print("E/N   <ε> (eV)          w (m/s)              D N (1/(m s))")
for reduced_field, (mean, drift, diffusion) in published.items():
    s = plasmacoll.solve_eedf(reid, [reid_gas], reduced_field)
    print(
        f"{reduced_field:3d}   {s.mean_energy:.4f} / {mean:.4f}   "
        f"{s.drift_velocity:.4g} / {drift:.4g}   {s.diffusion:.4g} / {diffusion:.4g}"
    )

The test suite also checks the Maxwell and Druyvesteyn distributions in their
limits, the Lucas–Saelee ionization model gas, that the discretisation loses no
electrons and closes the energy balance to second order in the cell size, and,
where [BOLOS](https://github.com/aluque/bolos) is installed, agreement with
BOLOS on the model gas above to better than 0.5 %.

The two-term approximation itself fails where the distribution is strongly
anisotropic, when inelastic collisions dominate the momentum transfer; multi-term
results for the Reid ramp differ by a few per cent in the drift velocity and by
about 20 % in the diffusion coefficient. For a plasma model's rates it is
usually accurate enough.